# Audit d'Équité — Modèle de Classification de CVs

**Objectif :** Évaluer si le modèle de classification `classification_cv_FINAL_500_a` traite les candidats de manière équitable, indépendamment de caractéristiques protégées.  

**Méthodologie :**
1. Identification des attributs sensibles et groupes protégés
2. Calcul des métriques d'équité (Demographic Parity, Equal Opportunity, Equalized Odds, Disparate Impact)
3. Analyse visuelle des disparités
4. Stratégies correctives (reweighting, threshold calibration, feature removal)
5. Explicabilité SHAP
6. Comparaison ancien vs nouveau modèle

**Cadre légal de référence :**  
- Directive européenne 2000/43/CE (non-discrimination)
- RGPD Art. 22 (décisions automatisées)
- AI Act (systèmes à haut risque : recrutement = Annexe III)
- Loi belge du 10 mai 2007 contre la discrimination


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score, ConfusionMatrixDisplay,
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

# Palette d'équité cohérente
FAIRNESS_COLORS = {'favorable': '#2ecc71', 'unfavorable': '#e74c3c', 'neutral': '#3498db'}

print('Imports OK ✓')

## 2. Chargement du Modèle et des Données

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/cv_dataset.csv')
print(f'Dataset : {df.shape[0]} candidats, {df.shape[1]} colonnes')
print(f'Taux de sélection global : {df["passed_next_stage"].mean():.1%}')

# Chargement du modèle sauvegardé
import os
model_path = '../models/model_classification_cv_cool.joblib'
if os.path.exists(model_path):
    model_data = joblib.load(model_path)
    lr_pipeline = model_data['pipeline']
    best_threshold = model_data['optimal_threshold']
    print(f'Modèle chargé ✓  |  Seuil optimal : {best_threshold:.4f}')
else:
    print('⚠ Modèle non trouvé — réentraînement requis (voir cellule suivante)')
    lr_pipeline = None
    best_threshold = 0.1434

In [ ]:
# ── Réentraînement si modèle absent ─────────────────────────────────────────
if lr_pipeline is None:
    df_tmp = df.copy()
    df_tmp['avg_gap_duration'] = df_tmp['total_gap_months'] / (df_tmp['nb_gaps'] + 1)
    df_tmp['gap_ratio'] = df_tmp['total_gap_months'] / (
        df_tmp['total_experience_years'] * 12 + df_tmp['total_gap_months'] + 1)
    df_tmp['skills_count'] = df_tmp['skills'].fillna('').apply(
        lambda x: len([s.strip() for s in x.split(',') if s.strip()]))
    df_tmp['certif_count'] = df_tmp['certifications'].fillna('').apply(
        lambda x: len([c.strip() for c in x.split(',') if c.strip()]))
    df_tmp['has_certif'] = (df_tmp['certif_count'] > 0).astype(int)

    NUMERIC_FEATURES = [
        'age', 'distance_ville_haute_km', 'total_experience_years',
        'nb_gaps', 'avg_gap_duration', 'gap_ratio', 'education_score',
        'skills_count', 'certif_count', 'has_certif',
        'lang_fr', 'lang_en', 'lang_de', 'lang_es', 'lang_it', 'lang_other_score_sum',
    ]
    CATEGORICAL_FEATURES = ['target_role', 'education_degree', 'education_field']
    TEXT_SKILLS = 'skills'
    TEXT_CERTIFICATIONS = 'certifications'

    TARGET = 'passed_next_stage'
    X_tmp = df_tmp.drop(columns=['cv_id', TARGET])
    y_tmp = df_tmp[TARGET]

    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUMERIC_FEATURES),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_FEATURES),
        ('skills', TfidfVectorizer(max_features=50, token_pattern=r'[a-zA-Z0-9#+\-\.]+', min_df=1), TEXT_SKILLS),
        ('certif', TfidfVectorizer(max_features=30, token_pattern=r'[a-zA-Z0-9#+\-\.]+', min_df=2), TEXT_CERTIFICATIONS),
    ], remainder='drop')

    lr_pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegressionCV(
            Cs=10, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
            penalty='l1', solver='liblinear', scoring='average_precision',
            class_weight={0: 1, 1: 0.5}, max_iter=1000, random_state=RANDOM_STATE,
        )),
    ])

    X_tr, X_te, y_tr, y_te = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tmp)
    for col in [TEXT_SKILLS, TEXT_CERTIFICATIONS]:
        X_tr[col] = X_tr[col].fillna('').str.lower().str.replace(r'[^\w\s,]', '', regex=True)
        X_te[col] = X_te[col].fillna('').str.lower().str.replace(r'[^\w\s,]', '', regex=True)

    lr_pipeline.fit(X_tr, y_tr)
    y_proba_te = lr_pipeline.predict_proba(X_te)[:, 1]
    precs, recs, thrs = precision_recall_curve(y_te, y_proba_te)
    beta = 0.5
    fbeta = (1 + beta**2) * (precs * recs) / (beta**2 * precs + recs + 1e-9)
    best_threshold = thrs[np.argmax(fbeta[:-1])]
    print(f'Modèle réentraîné  |  Seuil : {best_threshold:.4f}')

## 3. Feature Engineering & Prédictions sur le Dataset Complet

In [ ]:
# Feature engineering identique au notebook de classification
df_audit = df.copy()

df_audit['avg_gap_duration'] = df_audit['total_gap_months'] / (df_audit['nb_gaps'] + 1)
df_audit['gap_ratio'] = df_audit['total_gap_months'] / (
    df_audit['total_experience_years'] * 12 + df_audit['total_gap_months'] + 1)
df_audit['skills_count'] = df_audit['skills'].fillna('').apply(
    lambda x: len([s.strip() for s in x.split(',') if s.strip()]))
df_audit['certif_count'] = df_audit['certifications'].fillna('').apply(
    lambda x: len([c.strip() for c in x.split(',') if c.strip()]))
df_audit['has_certif'] = (df_audit['certif_count'] > 0).astype(int)

# Nettoyage texte
for col in ['skills', 'certifications']:
    df_audit[col] = df_audit[col].fillna('').str.lower().str.replace(r'[^\w\s,]', '', regex=True)

X_full = df_audit.drop(columns=['cv_id', 'passed_next_stage'])
y_full = df_audit['passed_next_stage']

# Prédictions
df_audit['proba'] = lr_pipeline.predict_proba(X_full)[:, 1]
df_audit['pred'] = (df_audit['proba'] >= best_threshold).astype(int)

print(f'Prédictions calculées sur {len(df_audit)} candidats')
print(f'Taux de sélection réel    : {y_full.mean():.1%}')
print(f'Taux de sélection prédit  : {df_audit["pred"].mean():.1%}')

## 4. Définition des Attributs Sensibles & Groupes Protégés

### Justification du choix des attributs

| Attribut | Risque | Base légale |
|---|---|---|
| `age` | Discrimination par âge (< 30 vs > 35 ans) | Directive 2000/78/CE, Loi belge 2007 |
| `distance_ville_haute_km` | Proxy géographique/socioéconomique | RGPD Art. 9 (origine géographique) |
| `lang_fr` / langues | Proxy de nationalité ou d'origine | Directive 2000/43/CE |
| `education_degree` | Discrimination socioéconomique indirecte | AI Act Annexe III |
| `education_school` | Biais de prestige des institutions | AI Act Art. 10 |

**Note :** Le dataset ne contient pas de genre ni de nationalité explicite, mais ces variables peuvent servir de **proxies**.

In [ ]:
# ── Création des groupes protégés ───────────────────────────────────────────

# Groupe 1 : Âge
df_audit['age_group'] = pd.cut(
    df_audit['age'],
    bins=[0, 29, 34, 100],
    labels=['Junior (≤29)', 'Mid (30-34)', 'Senior (≥35)']
)

# Groupe 2 : Proximité géographique
# distance_ville_haute_km — médiane ~4790 km (beaucoup sont très éloignés)
df_audit['geo_group'] = pd.cut(
    df_audit['distance_ville_haute_km'],
    bins=[0, 1000, 5000, 99999],
    labels=['Local (<1000km)', 'Régional (1-5000km)', 'International (>5000km)']
)

# Groupe 3 : Langue française (proxy d'intégration locale)
df_audit['fr_speaker'] = df_audit['lang_fr'].apply(
    lambda x: 'Francophone' if x >= 4 else 'Non-francophone'
)

# Groupe 4 : Niveau d'éducation
df_audit['edu_level'] = df_audit['education_degree'].apply(
    lambda x: 'Master+' if 'Master' in str(x) or 'PhD' in str(x) else 'Bachelor ou moins'
)

# Groupe 5 : Score éducation (école)
df_audit['edu_score_group'] = df_audit['education_score'].apply(
    lambda x: 'École de prestige (score 4)' if x >= 4 else 'École standard (score 3)'
)

print('Groupes définis :')
for col in ['age_group', 'geo_group', 'fr_speaker', 'edu_level', 'edu_score_group']:
    print(f'  {col} : {df_audit[col].value_counts().to_dict()}')

## 5. Métriques d'Équité — Définitions & Calcul

### Métriques choisies et justification

| Métrique | Formule | Interprétation | Seuil d'alerte |
|---|---|---|---|
| **Demographic Parity Difference** | P(Ŷ=1\|A=a) − P(Ŷ=1\|A=b) | Différence de taux de sélection entre groupes | > ±0.10 |
| **Disparate Impact Ratio** | P(Ŷ=1\|A=a) / P(Ŷ=1\|A=b) | Rapport des taux (règle des 80%) | < 0.80 |
| **Equal Opportunity Difference** | TPR(A=a) − TPR(B=b) | Différence de taux de vrais positifs | > ±0.10 |
| **Equalized Odds** | Combine EOD et FPR difference | Équité jointe sur TP et FP | > ±0.10 |
| **Predictive Parity** | PPV(A=a) − PPV(A=b) | Précision du modèle par groupe | > ±0.10 |

In [ ]:
def compute_fairness_metrics(df, group_col, y_true_col='passed_next_stage',
                              y_pred_col='pred', proba_col='proba'):
    """
    Calcule les métriques d'équité pour chaque groupe d'un attribut sensible.
    Retourne un DataFrame avec les métriques par groupe.
    """
    results = []
    groups = df[group_col].dropna().unique()

    for group in groups:
        mask = df[group_col] == group
        sub = df[mask]

        y_true = sub[y_true_col]
        y_pred = sub[y_pred_col]

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel() \
            if y_pred.sum() > 0 and (1 - y_pred).sum() > 0 \
            else (0, 0, 0, 0)

        n = len(sub)
        n_pos_true = y_true.sum()
        n_neg_true = n - n_pos_true

        selection_rate = y_pred.mean()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan  # Equal Opportunity
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan  # Précision
        accuracy = (tp + tn) / n if n > 0 else np.nan

        results.append({
            'group': group,
            'n': n,
            'n_pos_true': int(n_pos_true),
            'selection_rate': round(selection_rate, 4),
            'tpr': round(tpr, 4) if not np.isnan(tpr) else np.nan,
            'fpr': round(fpr, 4) if not np.isnan(fpr) else np.nan,
            'ppv': round(ppv, 4) if not np.isnan(ppv) else np.nan,
            'accuracy': round(accuracy, 4),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        })

    metrics_df = pd.DataFrame(results).set_index('group')

    # Demographic Parity Difference (vs groupe avec le taux max)
    max_sr = metrics_df['selection_rate'].max()
    min_sr = metrics_df['selection_rate'].min()
    metrics_df['dp_diff'] = metrics_df['selection_rate'] - max_sr
    metrics_df['disparate_impact'] = metrics_df['selection_rate'] / max_sr

    max_tpr = metrics_df['tpr'].max()
    metrics_df['eo_diff'] = metrics_df['tpr'] - max_tpr

    return metrics_df, max_sr - min_sr


print('Fonction compute_fairness_metrics définie ✓')

In [ ]:
# ── Calcul des métriques pour chaque attribut sensible ──────────────────────
sensitive_attrs = {
    'Âge (groupe)': 'age_group',
    'Distance géographique': 'geo_group',
    'Francophonie': 'fr_speaker',
    'Niveau éducation': 'edu_level',
    'Prestige école': 'edu_score_group',
    'Rôle visé': 'target_role',
}

all_metrics = {}
summary_rows = []

for label, col in sensitive_attrs.items():
    metrics, dp_gap = compute_fairness_metrics(df_audit, col)
    all_metrics[label] = metrics

    max_di = metrics['disparate_impact'].min()  # valeur la plus basse = plus discriminatoire
    alert = '🔴 ALERTE' if max_di < 0.80 or dp_gap > 0.10 else ('🟡 Attention' if dp_gap > 0.05 else '🟢 OK')

    summary_rows.append({
        'Attribut': label,
        'DP Gap (max-min)': round(dp_gap, 4),
        'Disparate Impact min': round(max_di, 4),
        'Statut': alert,
    })

summary_df = pd.DataFrame(summary_rows).set_index('Attribut')
print('=== RÉSUMÉ DE L\'AUDIT D\'ÉQUITÉ ===')
display(summary_df)

## 6. Visualisation des Disparités

In [ ]:
# ── Graphique 1 : Taux de sélection par groupe (tous attributs) ─────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

global_rate = df_audit['passed_next_stage'].mean()

for idx, (label, col) in enumerate(sensitive_attrs.items()):
    ax = axes[idx]
    metrics = all_metrics[label]
    sr = metrics['selection_rate'].sort_values()

    colors = ['#e74c3c' if v < global_rate * 0.8 else '#f39c12' if v < global_rate else '#2ecc71'
              for v in sr.values]

    bars = ax.barh(sr.index.astype(str), sr.values, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(global_rate, color='black', linestyle='--', lw=1.5, label=f'Moyenne ({global_rate:.1%})')
    ax.axvline(global_rate * 0.8, color='red', linestyle=':', lw=1, alpha=0.7, label='Seuil 80% (DI)')

    for bar, val in zip(bars, sr.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.1%}', va='center', fontsize=9, fontweight='bold')

    ax.set_xlim(0, max(sr.values) * 1.3)
    ax.set_title(f'Taux de sélection — {label}', fontweight='bold')
    ax.set_xlabel('Taux de sélection prédit')
    ax.legend(fontsize=8)

plt.suptitle('Audit d\'Équité : Taux de Sélection par Groupe', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/audit_selection_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée ✓')

In [ ]:
# ── Graphique 2 : Disparate Impact Ratio ────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (label, col) in enumerate(sensitive_attrs.items()):
    ax = axes[idx]
    metrics = all_metrics[label]
    di = metrics['disparate_impact'].sort_values()

    colors = ['#e74c3c' if v < 0.80 else '#f39c12' if v < 0.90 else '#2ecc71' for v in di.values]
    ax.barh(di.index.astype(str), di.values, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(1.0, color='black', linestyle='-', lw=1, alpha=0.5)
    ax.axvline(0.80, color='red', linestyle='--', lw=1.5, label='Seuil légal 80%')
    ax.axvline(0.90, color='orange', linestyle=':', lw=1, label='Seuil prudentiel 90%')

    ax.set_xlim(0, 1.2)
    ax.set_title(f'Disparate Impact — {label}', fontweight='bold')
    ax.set_xlabel('Ratio (1.0 = parité parfaite)')
    ax.legend(fontsize=8)

plt.suptitle('Disparate Impact Ratio par Groupe (< 0.80 = discrimination présumée)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/audit_disparate_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Graphique 3 : Equal Opportunity (TPR) & Equalized Odds ──────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (label, col) in enumerate(sensitive_attrs.items()):
    ax = axes[idx]
    metrics = all_metrics[label].dropna(subset=['tpr', 'fpr'])

    x = np.arange(len(metrics))
    w = 0.35
    ax.bar(x - w/2, metrics['tpr'], w, label='TPR (Equal Opp.)', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, metrics['fpr'], w, label='FPR (Equalized Odds)', color='coral', alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels([str(g)[:20] for g in metrics.index], rotation=20, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'TPR & FPR — {label}', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Equal Opportunity & Equalized Odds par Groupe', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/audit_equal_opportunity.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Analyse Détaillée par Attribut Sensible

In [ ]:
# ── 7.1 : Âge — analyse approfondie ─────────────────────────────────────────
print('=== ANALYSE : ÂGE ===')
print()
metrics_age = all_metrics['Âge (groupe)']
display(metrics_age[['n', 'n_pos_true', 'selection_rate', 'tpr', 'fpr', 'ppv', 'disparate_impact', 'Statut' if 'Statut' in metrics_age.columns else 'dp_diff']].round(3))

print()
print('Corrélation âge vs probabilité de sélection :')
corr = df_audit[['age', 'proba']].corr().iloc[0, 1]
print(f'  Pearson r = {corr:.3f}  ({"corrélation positive" if corr > 0 else "corrélation négative"})')

# Scatter âge vs proba
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_audit['age'], df_audit['proba'],
                c=df_audit['passed_next_stage'].map({0: '#e74c3c', 1: '#2ecc71'}),
                alpha=0.6, edgecolors='none')
axes[0].axhline(best_threshold, color='black', linestyle='--', lw=1.5, label=f'Seuil {best_threshold:.2f}')
axes[0].set_xlabel('Âge')
axes[0].set_ylabel('Probabilité de sélection')
axes[0].set_title('Âge vs Probabilité de sélection')
red_patch = mpatches.Patch(color='#e74c3c', label='Refusé (réel)')
green_patch = mpatches.Patch(color='#2ecc71', label='Sélectionné (réel)')
axes[0].legend(handles=[red_patch, green_patch])

sns.boxplot(data=df_audit, x='age_group', y='proba', ax=axes[1],
            palette='Set2', order=['Junior (≤29)', 'Mid (30-34)', 'Senior (≥35)'])
axes[1].axhline(best_threshold, color='red', linestyle='--', lw=1.5)
axes[1].set_title('Distribution des probabilités par groupe d\'âge')
axes[1].set_xlabel('Groupe d\'âge')
axes[1].set_ylabel('Probabilité prédite')

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 : Distance géographique ──────────────────────────────────────────────
print('=== ANALYSE : DISTANCE GÉOGRAPHIQUE ===')
display(all_metrics['Distance géographique'][['n', 'selection_rate', 'tpr', 'disparate_impact']].round(3))

print()
print('⚠ ATTENTION : La distance est utilisée comme feature directe dans le modèle.')
print('   Cela peut pénaliser les candidats éloignés (géographiquement défavorisés).')
print(f'   Corrélation distance vs proba : {df_audit[["distance_ville_haute_km", "proba"]].corr().iloc[0,1]:.3f}')

In [ ]:
# ── 7.3 : Rôle visé ──────────────────────────────────────────────────────────
print('=== ANALYSE : RÔLE VISÉ ===')
metrics_role = all_metrics['Rôle visé'].sort_values('selection_rate', ascending=False)
display(metrics_role[['n', 'selection_rate', 'tpr', 'fpr', 'disparate_impact']].round(3))

fig, ax = plt.subplots(figsize=(12, 6))
sr = metrics_role['selection_rate']
colors = ['#e74c3c' if v < df_audit['passed_next_stage'].mean() * 0.8 else '#2ecc71' for v in sr.values]
ax.barh(sr.index.astype(str), sr.values, color=colors, alpha=0.85)
ax.axvline(df_audit['passed_next_stage'].mean(), color='black', linestyle='--', lw=1.5)
ax.set_title('Taux de sélection prédit par rôle — Disparités inter-métiers', fontweight='bold')
ax.set_xlabel('Taux de sélection')
plt.tight_layout()
plt.show()

## 8. Explicabilité du Modèle — Coefficients L1 & Impact des Features Sensibles

L'explicabilité est une exigence de l'AI Act (Art. 13 : transparence) et du RGPD Art. 22.

In [ ]:
# ── Extraction des coefficients du modèle ────────────────────────────────────
preprocessor = lr_pipeline.named_steps['preprocessor']
classifier = lr_pipeline.named_steps['classifier']

NUMERIC_FEATURES = [
    'age', 'distance_ville_haute_km', 'total_experience_years',
    'nb_gaps', 'avg_gap_duration', 'gap_ratio', 'education_score',
    'skills_count', 'certif_count', 'has_certif',
    'lang_fr', 'lang_en', 'lang_de', 'lang_es', 'lang_it', 'lang_other_score_sum',
]
CATEGORICAL_FEATURES = ['target_role', 'education_degree', 'education_field']

ohe_names = (
    preprocessor.named_transformers_['cat']
    .named_steps['ohe']
    .get_feature_names_out(CATEGORICAL_FEATURES)
    .tolist()
)

skills_names = [
    f"skill_{w}" for w in
    preprocessor.named_transformers_['skills'].get_feature_names_out()
]
certif_names = [
    f"certif_{w}" for w in
    preprocessor.named_transformers_['certif'].get_feature_names_out()
]

feature_names = NUMERIC_FEATURES + ohe_names + skills_names + certif_names
coefs = classifier.coef_[0]

coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': coefs, 'abs_coef': np.abs(coefs)})
    .sort_values('abs_coef', ascending=False)
)

active_features = coef_df[coef_df['abs_coef'] > 0]
print(f'Features actives (L1) : {len(active_features)} / {len(coef_df)}')
print()
display(active_features.head(20))

In [ ]:
# ── Visualisation des coefficients avec highlighting des features sensibles ──
SENSITIVE_FEATURES = ['age', 'distance_ville_haute_km', 'lang_fr', 'lang_en', 'lang_de',
                      'lang_es', 'lang_it', 'lang_other_score_sum']

fig, ax = plt.subplots(figsize=(12, max(6, len(active_features) * 0.4)))

colors_coef = []
for _, row in active_features.iterrows():
    feat = row['feature']
    is_sensitive = any(s in feat for s in SENSITIVE_FEATURES)
    if is_sensitive:
        colors_coef.append('#e74c3c' if row['coef'] < 0 else '#f39c12')
    else:
        colors_coef.append('#2ecc71' if row['coef'] > 0 else '#3498db')

bars = ax.barh(
    active_features['feature'].values[::-1],
    active_features['coef'].values[::-1],
    color=colors_coef[::-1],
    alpha=0.85, edgecolor='white'
)
ax.axvline(0, color='black', lw=1)

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Positif neutre'),
    mpatches.Patch(color='#3498db', label='Négatif neutre'),
    mpatches.Patch(color='#f39c12', label='⚠ Feature SENSIBLE positive'),
    mpatches.Patch(color='#e74c3c', label='⚠ Feature SENSIBLE négative'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')
ax.set_title('Coefficients du modèle L1 — Identification des features sensibles', fontweight='bold')
ax.set_xlabel('Coefficient (impact sur la probabilité de sélection)')

plt.tight_layout()
plt.savefig('../data/audit_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Analyse de la contribution individuelle par feature sensible ─────────────
print('=== IMPACT DES FEATURES SENSIBLES DANS LE MODÈLE ===')
print()

sensitive_active = active_features[active_features['feature'].isin(SENSITIVE_FEATURES)]

if len(sensitive_active) > 0:
    for _, row in sensitive_active.iterrows():
        direction = 'FAVORISE' if row['coef'] > 0 else 'PÉNALISE'
        print(f"  • {row['feature']:<35} coef={row['coef']:+.4f}  → {direction} la sélection")
    print()
    print('⚠ ALERTE : Les features suivantes ont une influence directe dans le modèle')
    print('   et peuvent introduire une discrimination indirecte ou directe :')
    for feat in sensitive_active['feature'].tolist():
        print(f'   - {feat}')
else:
    print('Aucune feature sensible n\'a de coefficient non-nul (L1 les a éliminées).')

## 9. Stratégies Correctives

### Techniques implémentées

| Stratégie | Type | Principe |
|---|---|---|
| **Suppression features sensibles** | Pre-processing | Retirer `age` et `distance_ville_haute_km` |
| **Calibration du seuil par groupe** | Post-processing | Ajuster le seuil pour égaliser les TPR |
| **Reweighting des classes** | In-processing | Pondérer les exemples selon le groupe |
| **Contrainte d'équité (FairLearn)** | In-processing | Optimisation sous contrainte de parité |


In [ ]:
# ── Stratégie 1 : Suppression des features sensibles ────────────────────────
print('=== STRATÉGIE 1 : Modèle sans features directement sensibles ===')
print('Suppression de : age, distance_ville_haute_km')
print()

NUMERIC_FEATURES_FAIR = [
    # 'age',  ← SUPPRIMÉ
    # 'distance_ville_haute_km',  ← SUPPRIMÉ
    'total_experience_years',
    'nb_gaps', 'avg_gap_duration', 'gap_ratio', 'education_score',
    'skills_count', 'certif_count', 'has_certif',
    'lang_fr', 'lang_en', 'lang_de', 'lang_es', 'lang_it', 'lang_other_score_sum',
]

preprocessor_fair = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUMERIC_FEATURES_FAIR),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_FEATURES),
    ('skills', TfidfVectorizer(max_features=50, token_pattern=r'[a-zA-Z0-9#+\-\.]+', min_df=1), 'skills'),
    ('certif', TfidfVectorizer(max_features=30, token_pattern=r'[a-zA-Z0-9#+\-\.]+', min_df=2), 'certifications'),
], remainder='drop')

lr_fair = ImbPipeline([
    ('preprocessor', preprocessor_fair),
    ('classifier', LogisticRegressionCV(
        Cs=10, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        penalty='l1', solver='liblinear', scoring='average_precision',
        class_weight={0: 1, 1: 3},  # Rééquilibrage plus fort
        max_iter=1000, random_state=RANDOM_STATE,
    )),
])

# Train/test split
TARGET = 'passed_next_stage'
df_fe = df_audit.copy()
X_fe = df_fe.drop(columns=['cv_id', TARGET, 'proba', 'pred',
                            'age_group', 'geo_group', 'fr_speaker', 'edu_level', 'edu_score_group'])
y_fe = df_fe[TARGET]

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=RANDOM_STATE, stratify=y_fe)

lr_fair.fit(X_tr_f, y_tr_f)

# Seuil optimisé F0.5
y_proba_fair = lr_fair.predict_proba(X_te_f)[:, 1]
precs_f, recs_f, thrs_f = precision_recall_curve(y_te_f, y_proba_fair)
beta = 0.5
fbeta_f = (1 + beta**2) * (precs_f * recs_f) / (beta**2 * precs_f + recs_f + 1e-9)
threshold_fair = thrs_f[np.argmax(fbeta_f[:-1])]
y_pred_fair_te = (y_proba_fair >= threshold_fair).astype(int)

print(f'Seuil optimal modèle équitable : {threshold_fair:.4f}')
print()
print('Performances sur le jeu de test :')
print(classification_report(y_te_f, y_pred_fair_te, target_names=['Refusé', 'Sélectionné']))
print(f'ROC-AUC : {roc_auc_score(y_te_f, y_proba_fair):.3f}')

In [ ]:
# ── Stratégie 2 : Calibration du seuil par groupe (post-processing) ──────────
print('=== STRATÉGIE 2 : Calibration du seuil par groupe d\'âge (Equal Opportunity) ===')
print()

# Prédictions du modèle corrigé sur TOUT le dataset
df_fe2 = df_audit.copy()
X_all_fair = df_fe2.drop(columns=['cv_id', TARGET, 'proba', 'pred',
                                   'age_group', 'geo_group', 'fr_speaker', 'edu_level', 'edu_score_group'])
df_audit['proba_fair'] = lr_fair.predict_proba(X_all_fair)[:, 1]

# Pour chaque groupe d'âge, calibrer le seuil pour atteindre TPR ≈ global
target_tpr = 0.65  # TPR cible (global)

group_thresholds = {}
print(f'TPR cible : {target_tpr:.0%}')
print()

for group in df_audit['age_group'].dropna().unique():
    mask = df_audit['age_group'] == group
    sub = df_audit[mask]
    y_true_g = sub[TARGET]
    y_proba_g = sub['proba_fair']

    if y_true_g.sum() == 0:
        group_thresholds[group] = threshold_fair
        continue

    precs_g, recs_g, thrs_g = precision_recall_curve(y_true_g, y_proba_g)

    # Trouver le seuil qui approche le TPR cible
    best_thr = threshold_fair
    best_gap = float('inf')
    for thr, tpr_val in zip(thrs_g, recs_g[:-1]):
        gap = abs(tpr_val - target_tpr)
        if gap < best_gap:
            best_gap = gap
            best_thr = thr

    group_thresholds[group] = best_thr
    actual_tpr = (y_proba_g >= best_thr).astype(int)[y_true_g == 1].mean() if y_true_g.sum() > 0 else np.nan
    print(f'  {str(group):<20}  seuil={best_thr:.3f}  TPR_résultant={actual_tpr:.1%}')

print()
print('Seuils calibrés par groupe :', group_thresholds)

# Appliquer les seuils calibrés
def apply_group_threshold(row, group_col, thresholds, proba_col='proba_fair'):
    group = row[group_col]
    thr = thresholds.get(group, threshold_fair)
    return int(row[proba_col] >= thr)

df_audit['pred_calibrated'] = df_audit.apply(
    apply_group_threshold, axis=1,
    group_col='age_group', thresholds=group_thresholds
)

print()
print('Taux de sélection après calibration :')
print(df_audit.groupby('age_group')['pred_calibrated'].mean().round(3))

## 10. Comparaison Ancien vs Nouveau Modèle

In [ ]:
# ── Recalcul des métriques d'équité pour le nouveau modèle ──────────────────
df_audit['pred_fair'] = (df_audit['proba_fair'] >= threshold_fair).astype(int)

comparison_rows = []

for label, col in sensitive_attrs.items():
    # Ancien modèle
    m_old, dp_old = compute_fairness_metrics(df_audit, col, y_pred_col='pred')
    di_old = m_old['disparate_impact'].min()

    # Nouveau modèle
    m_new, dp_new = compute_fairness_metrics(df_audit, col, y_pred_col='pred_fair')
    di_new = m_new['disparate_impact'].min()

    comparison_rows.append({
        'Attribut': label,
        'DP Gap — Ancien': round(dp_old, 4),
        'DP Gap — Nouveau': round(dp_new, 4),
        'Δ DP Gap': round(dp_old - dp_new, 4),
        'DI min — Ancien': round(di_old, 4),
        'DI min — Nouveau': round(di_new, 4),
        'Amélioration DI': '✅' if di_new > di_old else '❌',
    })

comp_df = pd.DataFrame(comparison_rows).set_index('Attribut')
print('=== COMPARAISON ÉQUITÉ : ANCIEN vs NOUVEAU MODÈLE ===')
display(comp_df)

In [ ]:
# ── Comparaison des performances globales ────────────────────────────────────
X_te_fe_full = X_te_f

# Ancien modèle sur le test set
X_te_orig = df_audit.drop(columns=['cv_id', TARGET, 'proba', 'pred', 'proba_fair', 'pred_fair',
                                    'pred_calibrated', 'age_group', 'geo_group', 'fr_speaker',
                                    'edu_level', 'edu_score_group']).iloc[X_te_f.index]

try:
    y_proba_old_te = lr_pipeline.predict_proba(X_te_orig)[:, 1]
    y_pred_old_te = (y_proba_old_te >= best_threshold).astype(int)

    print('=== PERFORMANCES GLOBALES SUR LE JEU DE TEST ===')
    print()
    print('--- Ancien modèle ---')
    print(classification_report(y_te_f, y_pred_old_te, target_names=['Refusé', 'Sélectionné']))
    print(f'ROC-AUC : {roc_auc_score(y_te_f, y_proba_old_te):.3f}')

    print()
    print('--- Nouveau modèle (sans age, distance) ---')
    print(classification_report(y_te_f, y_pred_fair_te, target_names=['Refusé', 'Sélectionné']))
    print(f'ROC-AUC : {roc_auc_score(y_te_f, y_proba_fair):.3f}')
except Exception as e:
    print(f'Comparaison directe impossible (features différentes) — {e}')
    print()
    print('--- Nouveau modèle (sans age, distance) ---')
    print(classification_report(y_te_f, y_pred_fair_te, target_names=['Refusé', 'Sélectionné']))
    print(f'ROC-AUC : {roc_auc_score(y_te_f, y_proba_fair):.3f}')

In [ ]:
# ── Graphique de comparaison visuelle ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Taux de sélection par groupe d'âge
age_old = df_audit.groupby('age_group')['pred'].mean()
age_new = df_audit.groupby('age_group')['pred_fair'].mean()

x = np.arange(len(age_old))
w = 0.35
axes[0].bar(x - w/2, age_old.values, w, label='Ancien modèle', color='#e74c3c', alpha=0.8)
axes[0].bar(x + w/2, age_new.values, w, label='Nouveau modèle', color='#2ecc71', alpha=0.8)
axes[0].axhline(df_audit[TARGET].mean(), color='black', linestyle='--', lw=1.5, label='Taux réel')
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(g) for g in age_old.index], rotation=15)
axes[0].set_title('Taux de sélection par groupe d\'âge', fontweight='bold')
axes[0].set_ylabel('Taux de sélection')
axes[0].legend()
axes[0].set_ylim(0, 0.6)

# Disparate Impact
di_comparison = comp_df[['DI min — Ancien', 'DI min — Nouveau']]
x2 = np.arange(len(di_comparison))
axes[1].bar(x2 - w/2, di_comparison['DI min — Ancien'], w, label='Ancien modèle', color='#e74c3c', alpha=0.8)
axes[1].bar(x2 + w/2, di_comparison['DI min — Nouveau'], w, label='Nouveau modèle', color='#2ecc71', alpha=0.8)
axes[1].axhline(0.80, color='red', linestyle='--', lw=1.5, label='Seuil légal 80%')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(di_comparison.index, rotation=20, ha='right', fontsize=8)
axes[1].set_title('Disparate Impact minimum par attribut', fontweight='bold')
axes[1].set_ylabel('Disparate Impact (1.0 = parité)')
axes[1].legend()
axes[1].set_ylim(0, 1.3)

plt.suptitle('Comparaison Équité : Ancien vs Nouveau Modèle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/audit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Transparence — Explication Individuelle d'une Décision

In [ ]:
# ── Explication par décomposition des contributions de features (log-odds) ──
def explain_prediction_logistic(pipeline, sample_df, feature_names_num, feature_names_cat,
                                  text_cols=None, threshold=0.5):
    """
    Explique la décision du modèle logistique sur un candidat via les contributions log-odds.
    """
    clf = pipeline.named_steps['classifier']
    prep = pipeline.named_steps['preprocessor']

    X_transformed = prep.transform(sample_df)
    coefs = clf.coef_[0]
    intercept = clf.intercept_[0]

    contributions = X_transformed[0] * coefs
    log_odds = intercept + contributions.sum()
    proba = 1 / (1 + np.exp(-log_odds))

    ohe_names = prep.named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(feature_names_cat)
    skills_names_l = [f"skill_{w}" for w in prep.named_transformers_['skills'].get_feature_names_out()]
    certif_names_l = [f"certif_{w}" for w in prep.named_transformers_['certif'].get_feature_names_out()]
    all_feat = list(feature_names_num) + list(ohe_names) + skills_names_l + certif_names_l

    contrib_df = pd.DataFrame({
        'feature': all_feat,
        'contribution': contributions,
        'abs_contribution': np.abs(contributions),
    }).sort_values('abs_contribution', ascending=False)

    return proba, contrib_df[contrib_df['abs_contribution'] > 0], log_odds


# ── Exemple : expliquer la décision pour le 1er candidat ────────────────────
sample_idx = 0
sample = X_all_fair.iloc[[sample_idx]]
sample_info = df_audit.iloc[sample_idx]

proba_sample, contrib_sample, log_odds_sample = explain_prediction_logistic(
    lr_fair, sample,
    NUMERIC_FEATURES_FAIR, CATEGORICAL_FEATURES
)

decision = '✅ SÉLECTIONNÉ' if proba_sample >= threshold_fair else '❌ REFUSÉ'

print(f'=== EXPLICATION DE DÉCISION — Candidat {sample_info["cv_id"]} ===')
print(f'Rôle visé         : {sample_info["target_role"]}')
print(f'Expérience        : {sample_info["total_experience_years"]} ans')
print(f'Score éducation   : {sample_info["education_score"]}')
print()
print(f'Probabilité prédite : {proba_sample:.1%}')
print(f'Seuil de décision   : {threshold_fair:.1%}')
print(f'Décision            : {decision}')
print()
print('Top contributions (features qui ont le plus influencé la décision) :')
display(contrib_sample.head(10))

In [ ]:
# ── Graphique d'explication ──────────────────────────────────────────────────
top_n = min(10, len(contrib_sample))
top_contrib = contrib_sample.head(top_n).sort_values('contribution')

fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.5)))

colors_exp = ['#e74c3c' if v < 0 else '#2ecc71' for v in top_contrib['contribution']]
ax.barh(top_contrib['feature'], top_contrib['contribution'], color=colors_exp, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)

ax.set_title(
    f'Explication de la décision — {sample_info["cv_id"]}\n'
    f'Probabilité : {proba_sample:.1%}  |  Décision : {decision}',
    fontweight='bold'
)
ax.set_xlabel('Contribution au log-odds (>0 = favorise la sélection)')

plt.tight_layout()
plt.savefig('../data/audit_explanation_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Sauvegarde du Modèle Équitable

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

fair_model_data = {
    'pipeline': lr_fair,
    'optimal_threshold': threshold_fair,
    'group_thresholds': group_thresholds,
    'removed_features': ['age', 'distance_ville_haute_km'],
    'numeric_features': NUMERIC_FEATURES_FAIR,
    'categorical_features': CATEGORICAL_FEATURES,
    'audit_summary': summary_df.to_dict(),
    'fairness_metrics': {
        label: m.to_dict() for label, m in all_metrics.items()
    },
}

joblib.dump(fair_model_data, '../models/model_classification_cv_FAIR.joblib')
print(f'Modèle équitable sauvegardé ✓')
print(f'Seuil global     : {threshold_fair:.4f}')
print(f'Features retirées: age, distance_ville_haute_km')
print(f'Taille du modèle : {os.path.getsize("../models/model_classification_cv_FAIR.joblib") / 1024:.1f} KB')

## 13. Synthèse de l'Audit & Recommandations

### Conclusions principales

**Problèmes identifiés dans l'ancien modèle :**
1. **Utilisation directe de l'âge** comme feature — risque de discrimination par âge (Directive 2000/78/CE)
2. **Distance géographique** incluse — proxy potentiel d'origine nationale ou socioéconomique
3. **Scores de langues** (lang_fr, lang_de, etc.) avec coefficients actifs — risque de discrimination indirecte par nationalité
4. **Déséquilibre par rôle** — certains métiers ont un taux de sélection très différent

**Métriques d'équité de référence :**
- Demographic Parity Difference (DPD) : écart de taux de sélection entre groupes
- Disparate Impact Ratio (DIR) : règle des 80% (< 0.80 = discrimination présumée)
- Equal Opportunity : égalité des TPR entre groupes (candidats qualifiés traités équitablement)

**Mesures correctives appliquées :**
1. ✅ Suppression de `age` et `distance_ville_haute_km` du modèle corrigé
2. ✅ Calibration du seuil par groupe d'âge pour Equal Opportunity
3. ✅ Transparence via explication individuelle des décisions

**Recommandations complémentaires :**
- Effectuer un audit annuel avec des données fraîches
- Documenter les décisions contestées (RGPD Art. 22 — droit à l'explication)
- Envisager une boucle de feedback humain pour les cas limites (seuil ±5%)
- Collecter des données démographiques anonymisées pour un audit plus précis
- Former les recruteurs aux biais algorithmiques

### Apports de la conférence expert

Un expert en éthique algorithmique et droit de la discrimination soulignera typiquement :
- **La distinction discrimination directe vs indirecte** : même sans genre ou nationalité, un modèle peut discriminer via des proxies
- **L'importance du contexte légal belge/européen** : l'AI Act classe les outils RH en catégorie à haut risque
- **Le principe de minimisation des données** (RGPD) : ne pas utiliser l'âge ou la distance si non nécessaire
- **Le droit à l'explication** : tout candidat refusé automatiquement doit pouvoir demander une explication humaine


In [ ]:
# ── Tableau de bord final ────────────────────────────────────────────────────
print('='*70)
print(' RAPPORT D\'AUDIT D\'ÉQUITÉ — TABLEAU DE BORD FINAL')
print('='*70)
print()
print(f'Dataset analysé     : {len(df_audit)} candidats')
print(f'Taux sélection réel : {df_audit[TARGET].mean():.1%}')
print()
print('── ANCIEN MODÈLE (LR L1 avec age + distance) ──')
print(f'  Features sensibles actives : age (coef +), distance (coef +), langues')
print(f'  Seuil de décision          : {best_threshold:.4f}')
print()
print('── NOUVEAU MODÈLE (LR L1 ÉQUITABLE) ──')
print(f'  Features sensibles retirées: age, distance_ville_haute_km')
print(f'  Seuil de décision          : {threshold_fair:.4f}')
print()
print('── MÉTRIQUES D\'ÉQUITÉ ──')
display(summary_df)
print()
print('── COMPARAISON ÉQUITÉ ──')
display(comp_df)
print()
print('Modèle équitable sauvegardé : ../models/model_classification_cv_FAIR.joblib')
print('='*70)